In [2]:
#"Developer: Reload Window" 

import optuna
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split as tts 
from sklearn.preprocessing import StandardScaler
import pandas as pd 


In [3]:
url="https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
columns=['Pregnancies','Glucose','BloodPressure','SkinThickness','Insulin','BMI','DiabetesPedigreeFunction','Age','Outcome']

In [4]:
df=pd.read_csv(url,names=columns)

In [5]:
df.head(5)

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [6]:
print(df.isnull().sum())

Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64


In [7]:
import numpy as np 
cols_with_missing_vals=['Glucose','BloodPressure','SkinThickness','Insulin','BMI']
df[cols_with_missing_vals]=df[cols_with_missing_vals].replace(0,np.nan)

df.fillna(df.mean(),inplace=True)

print(df.isnull().sum())

Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64


In [8]:
X=df.iloc[:,:-1]
y=df.iloc[:,-1]

X_train,X_test,y_train,y_test=tts(X,y,test_size=0.2,random_state=42)

sc=StandardScaler()
X_train=sc.fit_transform(X_train)
X_test=sc.transform(X_test)

print(X_train.shape)
print(X_test.shape)

(614, 8)
(154, 8)


In [9]:
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestClassifier

def objective(trial):
    n_estimator=trial.suggest_int('n_estimator',50,200)
    max_depth=trial.suggest_int('max_depth',3,20)

    model=RandomForestClassifier(
        n_estimators=n_estimator,
        max_depth=max_depth,
        random_state=42
    )

    score=cross_val_score(model,X_train,y_train,cv=3,scoring='accuracy').mean()
    return score

In [10]:
# Creating a study 
study=optuna.create_study(direction='maximize',sampler=optuna.samplers.TPESampler())
study.optimize(objective,n_trials=50)

[I 2025-12-28 13:35:30,674] A new study created in memory with name: no-name-b5f998af-02eb-4aca-9492-e240c678d44c
[I 2025-12-28 13:35:30,999] Trial 0 finished with value: 0.7752431053722302 and parameters: {'n_estimator': 68, 'max_depth': 15}. Best is trial 0 with value: 0.7752431053722302.
[I 2025-12-28 13:35:31,338] Trial 1 finished with value: 0.7752351347042882 and parameters: {'n_estimator': 89, 'max_depth': 16}. Best is trial 0 with value: 0.7752431053722302.
[I 2025-12-28 13:35:31,977] Trial 2 finished with value: 0.7703491152558585 and parameters: {'n_estimator': 179, 'max_depth': 8}. Best is trial 0 with value: 0.7752431053722302.
[I 2025-12-28 13:35:32,480] Trial 3 finished with value: 0.7784951378925554 and parameters: {'n_estimator': 134, 'max_depth': 17}. Best is trial 3 with value: 0.7784951378925554.
[I 2025-12-28 13:35:33,038] Trial 4 finished with value: 0.7784791965566714 and parameters: {'n_estimator': 150, 'max_depth': 15}. Best is trial 3 with value: 0.778495137892

In [16]:
print('best trial',study.best_trial.value)
print('best params',study.best_trial.params)

best trial 0.7866331898613104
best params {'n_estimator': 160, 'max_depth': 13}


In [22]:
from sklearn.metrics import accuracy_score
# Fix the wrong parameter name
best_params = study.best_params.copy()
best_params['n_estimators'] = best_params.pop('n_estimator')
bst_model=RandomForestClassifier(
    **best_params,
    random_state=42
)

bst_model.fit(X_train,y_train)
y_pred=bst_model.predict(X_test)

tst_acuracy=accuracy_score(y_test,y_pred)

print(tst_acuracy)

0.7532467532467533
